In [ ]:
# ============================================================
# УПРОЩЕННЫЙ CLTV NOTEBOOK - СРАВНЕНИЕ CATBOOST И LIGHTGBM
# Часть 1: Импорты, конфигурация, загрузка данных
# ============================================================

# %% ИМПОРТЫ И НАСТРОЙКИ
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from dateutil.relativedelta import relativedelta
import os
import warnings
from collections import defaultdict, deque
from scipy import stats
import logging
import pickle
import json
from pathlib import Path

from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")

print("Импорты загружены")

# %% КОНФИГУРАЦИЯ
class Config:
    TRAIN_PATH = "CLTV_UL_TRAIN_MART.csv"
    MODEL_DIR = Path("models")
    MODEL_VERSION = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    VALIDATION_CUTOFF = "2025-03-31"
    MIN_SAMPLES_PER_SEGMENT = 1000
    
    CATEGORICAL_FEATURES = ['QUALITY_CODE', 'SUBJECT_KIND_ID', 'EC_SECTOR_ID']
    BASE_FEATURES = [
        'MARGIN', 'MARGIN_LAG1', 'MARGIN_LAG2', 'MARGIN_LAG3',
        'MARGIN_AVG_1M_LAG', 'MARGIN_AVG_2M_LAG', 'MARGIN_AVG_3M_LAG',
        'MARGIN_AVG_6M_LAG', 'MARGIN_AVG_12M_LAG', 'MARGIN_STDDEV_12M_LAG',
        'MARGIN_GROWTH_RATE_3M', 'MONTH_OF_YEAR', 'QUARTER_OF_YEAR', 'TENURE_MONTHS'
    ]
    
    @classmethod
    def ensure_directories(cls):
        cls.MODEL_DIR.mkdir(parents=True, exist_ok=True)
        (cls.MODEL_DIR / cls.MODEL_VERSION).mkdir(parents=True, exist_ok=True)

Config.ensure_directories()
print(f"Версия: {Config.MODEL_VERSION}")

# %% ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
def read_table(path):
    if not os.path.exists(path):
        logger.warning(f"Файл не найден: {path}")
        return pd.DataFrame()
    ext = os.path.splitext(path)[1].lower()
    if ext in [".parquet", ".pq", ".parq"]:
        return pd.read_parquet(path)
    elif ext in [".csv", ".txt"]:
        try:
            return pd.read_csv(path, sep='|', encoding="windows-1251", thousands=',')
        except:
            try:
                return pd.read_csv(path, sep=',')
            except:
                return pd.read_csv(path, sep='\t')
    raise ValueError(f"Неподдерживаемый формат: {ext}")

def fix_categorical_features(df, cat_features):
    df_fixed = df.copy()
    for col in cat_features:
        if col in df_fixed.columns:
            df_fixed[col] = df_fixed[col].fillna('UNKNOWN').astype(str)
            df_fixed[col] = df_fixed[col].str.replace('.0', '', regex=False)
    return df_fixed

def stabilize_target(y):
    return np.sign(y) * np.log1p(np.abs(y))

def inverse_stabilize_target(y_stable):
    return np.sign(y_stable) * (np.exp(np.abs(y_stable)) - 1)

print("Функции загружены")

# %% ЗАГРУЗКА ДАННЫХ
logger.info("Загрузка данных...")
train = read_table(Config.TRAIN_PATH)

print(f"Обучающая: {len(train):,} записей, {train['CLIENT_ID'].nunique():,} клиентов")

# %% ПРЕДОБРАБОТКА
all_categorical = Config.CATEGORICAL_FEATURES + ['SEGMENT_ID']
train_fixed = fix_categorical_features(train, all_categorical)

numeric_features = [f for f in Config.BASE_FEATURES if f not in Config.CATEGORICAL_FEATURES]
for col in numeric_features:
    train_fixed[col] = train_fixed[col].fillna(0.0)

print(f"Фичи готовы")

In [ ]:
# ========== ГРУППИРОВКА СЕГМЕНТОВ ==========
print("\nГруппировка сегментов...")

def map_segment_groups(df):
    """
    Объединяем сегменты для более устойчивого моделирования:
    - 1026 (MICRO) + 1027 (SMALL) → 'SMB'
    - 1022 (MIDDLE) + 1023 (LARGE) → 'CORPORATE'
    """
    segment_mapping = {
        '1026': 'SMB',
        '1027': 'SMB',
        '1022': 'CORPORATE',
        '1023': 'CORPORATE',
        '1028': 'OTHER',
        '1040': 'OTHER'
    }
    df['SEGMENT_GROUP'] = df['SEGMENT_ID'].astype(str).map(segment_mapping).fillna('OTHER')
    return df

train_fixed = map_segment_groups(train_fixed)

print("Распределение по новым сегментам:")
print(train_fixed.groupby('SEGMENT_GROUP').agg({
    'CLIENT_ID': 'nunique',
    'SEGMENT_ID': 'count'
}).rename(columns={'CLIENT_ID': 'Клиентов', 'SEGMENT_ID': 'Записей'}))

# Обновить ALL_FEATURES (заменить 'SEGMENT_ID' на 'SEGMENT_GROUP')
ALL_FEATURES = ['SEGMENT_GROUP'] + Config.BASE_FEATURES + Config.CATEGORICAL_FEATURES
available_features = [f for f in ALL_FEATURES if f in train_fixed.columns]

print(f"\nФичей после группировки: {len(available_features)}")

In [ ]:
# ========== OUTLIER HANDLING ==========
from scipy.stats import mstats

print("\nПрименение Winsorization для MARGIN...")

# Колонки для обработки
margin_cols = ['MARGIN', 'MARGIN_LAG1', 'MARGIN_LAG2', 'MARGIN_LAG3',
               'MARGIN_AVG_1M_LAG', 'MARGIN_AVG_2M_LAG', 'MARGIN_AVG_3M_LAG',
               'MARGIN_AVG_6M_LAG', 'MARGIN_AVG_12M_LAG']

# Статистика до обработки
print(f"MARGIN до winsorization:")
print(f"  Min: {train_fixed['MARGIN'].min():,.0f}")
print(f"  Max: {train_fixed['MARGIN'].max():,.0f}")
print(f"  Std: {train_fixed['MARGIN'].std():,.0f}")

# Применяем winsorization (обрезаем 1% экстремумов с каждой стороны)
for col in margin_cols:
    if col in train_fixed.columns:
        train_fixed[col] = mstats.winsorize(train_fixed[col], limits=[0.01, 0.01])

print(f"\nMARGIN после winsorization:")
print(f"  Min: {train_fixed['MARGIN'].min():,.0f}")
print(f"  Max: {train_fixed['MARGIN'].max():,.0f}")
print(f"  Std: {train_fixed['MARGIN'].std():,.0f}")

# Добавляем бинарный флаг для отрицательных значений
train_fixed['IS_NEGATIVE_MARGIN'] = (train_fixed['MARGIN'] < 0).astype(int)
print(f"\nЗаписей с отрицательным margin: {train_fixed['IS_NEGATIVE_MARGIN'].sum():,} ({100*train_fixed['IS_NEGATIVE_MARGIN'].mean():.1f}%)")

# Обновляем список фичей
ALL_FEATURES = ['SEGMENT_GROUP', 'IS_NEGATIVE_MARGIN'] + Config.BASE_FEATURES + Config.CATEGORICAL_FEATURES
available_features = [f for f in ALL_FEATURES if f in train_fixed.columns]
print(f"Финальное количество фичей: {len(available_features)}")

In [ ]:
# ========== РАСШИРЕННЫЕ МЕТРИКИ ==========

def smape(y_true, y_pred):
    """Symmetric MAPE - устойчивая процентная метрика"""
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    diff = np.abs(y_true - y_pred)
    mask = denominator > 0
    return 100 * np.mean(diff[mask] / denominator[mask]) if mask.sum() > 0 else 0.0

def weighted_mae(y_true, y_pred):
    """MAE взвешенный по величине клиента"""
    weights = np.abs(y_true)
    if weights.sum() == 0:
        return mean_absolute_error(y_true, y_pred)
    weights = weights / weights.sum()
    return np.sum(weights * np.abs(y_true - y_pred))

def weighted_r2(y_true, y_pred):
    """R² взвешенный по важности клиента"""
    weights = np.abs(y_true)
    if weights.sum() == 0:
        return r2_score(y_true, y_pred)
    weights = weights / weights.sum()
    
    ss_res = np.sum(weights * (y_true - y_pred)**2)
    ss_tot = np.sum(weights * (y_true - np.average(y_true, weights=weights))**2)
    
    return 1 - (ss_res / ss_tot) if ss_tot > 0 else 0.0

def capture_rate(y_true, y_pred, top_n=0.1):
    """Процент value в top-N% predicted клиентов"""
    n_top = max(1, int(len(y_true) * top_n))
    top_predicted_idx = np.argsort(y_pred)[-n_top:]
    
    captured = y_true[top_predicted_idx].sum()
    total = y_true.sum()
    
    return (captured / total * 100) if total != 0 else 0.0

def comprehensive_metrics(y_true, y_pred):
    """Полный набор метрик для отчетности"""
    return {
        'R²': r2_score(y_true, y_pred),
        'MAE_млн': mean_absolute_error(y_true, y_pred) / 1e6,
        'RMSE_млн': np.sqrt(mean_squared_error(y_true, y_pred)) / 1e6,
        'MedAE_млн': np.median(np.abs(y_true - y_pred)) / 1e6,
        'sMAPE_%': smape(y_true, y_pred),
        'Weighted_R²': weighted_r2(y_true, y_pred),
        'Weighted_MAE_млн': weighted_mae(y_true, y_pred) / 1e6,
        'Capture_Rate_10%': capture_rate(y_true, y_pred, 0.1),
        'Capture_Rate_20%': capture_rate(y_true, y_pred, 0.2),
        'Sign_Accuracy_%': (np.sign(y_true) == np.sign(y_pred)).mean() * 100
    }

print("Функции метрик загружены")

In [ ]:
# ========== УПРОЩЕННАЯ МОДЕЛЬ CATBOOST ==========

class SimplifiedCatBoostCLTV:
    """Упрощенная версия без Optuna, churn, прогнозирования"""
    
    def __init__(self, min_samples_per_segment=1000):
        self.models = {}
        self.segment_stats = {}
        self.min_samples_per_segment = min_samples_per_segment
        self.fallback_segments = {}
        self.feature_importance = {}
        self.best_params = {}
        
    def analyze_segment_distribution(self, df, segment_col='SEGMENT_GROUP'):
        """Анализ распределения по сегментам"""
        segment_stats = df.groupby(segment_col).agg({
            'CLIENT_ID': 'nunique',
            'TARGET_NEXT_MARGIN': ['count', 'mean', 'std', 'min', 'max']
        }).round(2)
        
        segment_stats.columns = ['unique_clients', 'total_records', 'avg_margin', 
                               'std_margin', 'min_margin', 'max_margin']
        
        large = segment_stats[segment_stats['total_records'] >= self.min_samples_per_segment].index.tolist()
        small = segment_stats[segment_stats['total_records'] < self.min_samples_per_segment].index.tolist()
        
        print("Анализ сегментов:")
        print(segment_stats)
        print(f"\nБольшие сегменты (>={self.min_samples_per_segment}): {large}")
        print(f"Малые сегменты (<{self.min_samples_per_segment}): {small}")
        
        return segment_stats, large, small
    
    def get_default_params(self, segment_id, data_size):
        """Фиксированные параметры без оптимизации"""
        base = {
            "random_seed": 42,
            "loss_function": "MAE",
            "verbose": False,
            "early_stopping_rounds": 50
        }
        
        if data_size > 100000:
            base.update({
                "iterations": 2000,
                "depth": 6,
                "learning_rate": 0.01,
                "l2_leaf_reg": 20
            })
        elif data_size > 10000:
            base.update({
                "iterations": 1500,
                "depth": 5,
                "learning_rate": 0.015,
                "l2_leaf_reg": 30
            })
        else:
            base.update({
                "iterations": 1000,
                "depth": 4,
                "learning_rate": 0.02,
                "l2_leaf_reg": 50
            })
        
        return base
    
    def prepare_segment_data(self, df, segment_id, features, target_col, validation_cutoff):
        """Подготовка данных сегмента"""
        segment_data = df[df['SEGMENT_GROUP'] == segment_id].copy()
        segment_data['target_stable'] = stabilize_target(segment_data[target_col])
        
        train_mask = pd.to_datetime(segment_data['MONTH_END']) <= pd.to_datetime(validation_cutoff)
        val_mask = ~train_mask
        
        features_for_segment = [f for f in features if f != 'SEGMENT_GROUP']
        
        X_train = segment_data[train_mask][features_for_segment]
        y_train = segment_data[train_mask]['target_stable']
        X_val = segment_data[val_mask][features_for_segment]
        y_val = segment_data[val_mask]['target_stable']
        y_val_original = segment_data[val_mask][target_col]
        
        return X_train, y_train, X_val, y_val, y_val_original
    
    def train_segment_model(self, segment_id, X_train, y_train, X_val, y_val, 
                           y_val_original, categorical_features=None):
        """Обучение модели для сегмента"""
        
        cat_indices = []
        if categorical_features:
            features_list = X_train.columns.tolist()
            for cat_feat in categorical_features:
                if cat_feat in features_list:
                    cat_indices.append(features_list.index(cat_feat))
        
        # Получаем параметры
        params = self.get_default_params(segment_id, len(X_train))
        self.best_params[segment_id] = params
        
        print(f"  Параметры: iterations={params['iterations']}, depth={params['depth']}, lr={params['learning_rate']}")
        
        # Создаем пулы
        train_pool = Pool(X_train, y_train, cat_features=cat_indices)
        val_pool = Pool(X_val, y_val, cat_features=cat_indices)
        
        # Обучаем
        model = CatBoostRegressor(**params)
        model.fit(train_pool, eval_set=val_pool, use_best_model=True)
        
        # Предсказания
        train_pred_stable = model.predict(X_train)
        val_pred_stable = model.predict(X_val)
        
        # Обратная трансформация
        val_pred_original = inverse_stabilize_target(val_pred_stable)
        
        # Метрики на stable scale
        metrics = {
            'train_r2_stable': r2_score(y_train, train_pred_stable),
            'val_r2_stable': r2_score(y_val, val_pred_stable),
            'val_mae_stable': mean_absolute_error(y_val, val_pred_stable),
            'train_samples': len(X_train),
            'val_samples': len(X_val)
        }
        
        # Комплексные метрики на original scale
        comprehensive = comprehensive_metrics(y_val_original, val_pred_original)
        metrics.update(comprehensive)
        
        # Feature importance
        metrics['feature_importance'] = pd.DataFrame({
            'feature': X_train.columns,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        return model, metrics

print("Класс SimplifiedCatBoostCLTV создан")

In [ ]:
# ========== LIGHTGBM ДЛЯ СРАВНЕНИЯ ==========

import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

class LightGBMCLTV:
    """LightGBM версия для сравнения"""
    
    def __init__(self, min_samples_per_segment=1000):
        self.models = {}
        self.segment_stats = {}
        self.min_samples_per_segment = min_samples_per_segment
        self.fallback_segments = {}
        self.feature_importance = {}
        self.best_params = {}
        self.label_encoders = {}
        
    def get_lgb_params(self, data_size):
        """Фиксированные параметры LightGBM"""
        base = {
            'objective': 'mae',
            'metric': 'mae',
            'boosting_type': 'gbdt',
            'verbose': -1,
            'seed': 42,
            'feature_pre_filter': False
        }
        
        if data_size > 100000:
            base.update({
                'num_leaves': 31,
                'max_depth': 6,
                'learning_rate': 0.01,
                'n_estimators': 2000,
                'min_child_samples': 20
            })
        else:
            base.update({
                'num_leaves': 20,
                'max_depth': 5,
                'learning_rate': 0.015,
                'n_estimators': 1500,
                'min_child_samples': 30
            })
        
        return base
    
    def prepare_segment_data(self, df, segment_id, features, target_col, validation_cutoff):
        """Подготовка данных сегмента"""
        segment_data = df[df['SEGMENT_GROUP'] == segment_id].copy()
        segment_data['target_stable'] = stabilize_target(segment_data[target_col])
        
        train_mask = pd.to_datetime(segment_data['MONTH_END']) <= pd.to_datetime(validation_cutoff)
        val_mask = ~train_mask
        
        features_for_segment = [f for f in features if f != 'SEGMENT_GROUP']
        
        X_train = segment_data[train_mask][features_for_segment]
        y_train = segment_data[train_mask]['target_stable']
        X_val = segment_data[val_mask][features_for_segment]
        y_val = segment_data[val_mask]['target_stable']
        y_val_original = segment_data[val_mask][target_col]
        
        return X_train, y_train, X_val, y_val, y_val_original
    
    def train_segment_model(self, segment_id, X_train, y_train, X_val, y_val, 
                           y_val_original, categorical_features=None):
        """Обучение LightGBM"""
        
        X_train_encoded = X_train.copy()
        X_val_encoded = X_val.copy()
        
        # Label encoding для категориальных
        if categorical_features:
            for cat_feat in categorical_features:
                if cat_feat in X_train.columns:
                    le = LabelEncoder()
                    X_train_encoded[cat_feat] = le.fit_transform(X_train[cat_feat].astype(str))
                    X_val_encoded[cat_feat] = le.transform(X_val[cat_feat].astype(str))
                    self.label_encoders[f"{segment_id}_{cat_feat}"] = le
        
        params = self.get_lgb_params(len(X_train))
        self.best_params[segment_id] = params
        
        print(f"  LightGBM: n_estimators={params['n_estimators']}, max_depth={params['max_depth']}")
        
        model = lgb.LGBMRegressor(**params)
        
        model.fit(
            X_train_encoded, y_train,
            eval_set=[(X_val_encoded, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(0)]
        )
        
        # Предсказания
        train_pred_stable = model.predict(X_train_encoded)
        val_pred_stable = model.predict(X_val_encoded)
        val_pred_original = inverse_stabilize_target(val_pred_stable)
        
        metrics = {
            'train_r2_stable': r2_score(y_train, train_pred_stable),
            'val_r2_stable': r2_score(y_val, val_pred_stable),
            'val_mae_stable': mean_absolute_error(y_val, val_pred_stable),
            'train_samples': len(X_train),
            'val_samples': len(X_val)
        }
        
        comprehensive = comprehensive_metrics(y_val_original, val_pred_original)
        metrics.update(comprehensive)
        
        metrics['feature_importance'] = pd.DataFrame({
            'feature': X_train.columns,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        return model, metrics

print("Класс LightGBMCLTV создан")

In [ ]:
# ========== ФУНКЦИЯ ОБУЧЕНИЯ ==========

def train_both_models(df, features, validation_cutoff, categorical_features):
    """Обучение CatBoost и LightGBM для сравнения"""
    
    results = {'catboost': {}, 'lightgbm': {}}
    
    # Инициализация моделей
    catboost_model = SimplifiedCatBoostCLTV(min_samples_per_segment=Config.MIN_SAMPLES_PER_SEGMENT)
    lightgbm_model = LightGBMCLTV(min_samples_per_segment=Config.MIN_SAMPLES_PER_SEGMENT)
    
    # Анализ сегментов
    segment_stats, large_segments, small_segments = catboost_model.analyze_segment_distribution(df)
    
    print(f"\n{'='*70}")
    print("ОБУЧЕНИЕ МОДЕЛЕЙ")
    print(f"{'='*70}\n")
    
    # Обучаем на больших сегментах
    for segment_id in large_segments:
        print(f"\n{'='*70}")
        print(f"Сегмент: {segment_id}")
        print(f"{'='*70}")
        
        try:
            # Подготовка данных
            X_train, y_train, X_val, y_val, y_val_orig = catboost_model.prepare_segment_data(
                df, segment_id, features, 'TARGET_NEXT_MARGIN', validation_cutoff
            )
            
            if len(X_train) < 100:
                print(f"  Пропуск: недостаточно данных ({len(X_train)})")
                continue
            
            # === CATBOOST ===
            print("\n[1] CatBoost:")
            cb_model, cb_metrics = catboost_model.train_segment_model(
                segment_id, X_train, y_train, X_val, y_val, y_val_orig, categorical_features
            )
            
            catboost_model.models[segment_id] = cb_model
            catboost_model.segment_stats[segment_id] = cb_metrics
            catboost_model.feature_importance[segment_id] = cb_metrics['feature_importance']
            
            print(f"    R²: {cb_metrics['R²']:.3f} | MAE: {cb_metrics['MAE_млн']:.1f}M | sMAPE: {cb_metrics['sMAPE_%']:.1f}%")
            
            # === LIGHTGBM ===
            print("\n[2] LightGBM:")
            lgb_model, lgb_metrics = lightgbm_model.train_segment_model(
                segment_id, X_train, y_train, X_val, y_val, y_val_orig, categorical_features
            )
            
            lightgbm_model.models[segment_id] = lgb_model
            lightgbm_model.segment_stats[segment_id] = lgb_metrics
            lightgbm_model.feature_importance[segment_id] = lgb_metrics['feature_importance']
            
            print(f"    R²: {lgb_metrics['R²']:.3f} | MAE: {lgb_metrics['MAE_млн']:.1f}M | sMAPE: {lgb_metrics['sMAPE_%']:.1f}%")
            
            # Сравнение
            winner = 'CatBoost' if cb_metrics['R²'] > lgb_metrics['R²'] else 'LightGBM'
            print(f"\n  → Лучше: {winner}")
            
            results['catboost'][segment_id] = cb_metrics
            results['lightgbm'][segment_id] = lgb_metrics
            
        except Exception as e:
            print(f"  Ошибка: {e}")
            continue
    
    return catboost_model, lightgbm_model, results

print("Функция train_both_models готова")

In [ ]:
# ========== ЗАПУСК ОБУЧЕНИЯ ==========

logger.info("Запуск обучения обеих моделей...")

catboost_cltv, lightgbm_cltv, training_results = train_both_models(
    df=train_fixed,
    features=available_features,
    validation_cutoff=Config.VALIDATION_CUTOFF,
    categorical_features=Config.CATEGORICAL_FEATURES
)

# ========== СВОДКА РЕЗУЛЬТАТОВ ==========

print("\n" + "="*100)
print("ИТОГОВОЕ СРАВНЕНИЕ МОДЕЛЕЙ")
print("="*100)

comparison_data = []
for segment_id in training_results['catboost'].keys():
    cb = training_results['catboost'][segment_id]
    lgb = training_results['lightgbm'][segment_id]
    
    comparison_data.append({
        'Segment': segment_id,
        'Samples': f"{cb['train_samples']:,} / {cb['val_samples']:,}",
        'CB_R²': f"{cb['R²']:.3f}",
        'LGB_R²': f"{lgb['R²']:.3f}",
        'CB_MAE_млн': f"{cb['MAE_млн']:.1f}",
        'LGB_MAE_млн': f"{lgb['MAE_млн']:.1f}",
        'CB_sMAPE': f"{cb['sMAPE_%']:.1f}",
        'LGB_sMAPE': f"{lgb['sMAPE_%']:.1f}",
        'CB_Capture10%': f"{cb['Capture_Rate_10%']:.1f}",
        'LGB_Capture10%': f"{lgb['Capture_Rate_10%']:.1f}"
    })

if comparison_data:
    df_comparison = pd.DataFrame(comparison_data)
    print(df_comparison.to_string(index=False))
    
    # Средние метрики
    print("\n" + "="*100)
    print("СРЕДНИЕ МЕТРИКИ")
    print("="*100)
    
    cb_avg_r2 = np.mean([m['R²'] for m in training_results['catboost'].values()])
    lgb_avg_r2 = np.mean([m['R²'] for m in training_results['lightgbm'].values()])
    
    cb_avg_mae = np.mean([m['MAE_млн'] for m in training_results['catboost'].values()])
    lgb_avg_mae = np.mean([m['MAE_млн'] for m in training_results['lightgbm'].values()])
    
    cb_avg_smape = np.mean([m['sMAPE_%'] for m in training_results['catboost'].values()])
    lgb_avg_smape = np.mean([m['sMAPE_%'] for m in training_results['lightgbm'].values()])
    
    print(f"\nCatBoost:")
    print(f"  Средний R²: {cb_avg_r2:.3f}")
    print(f"  Средний MAE: {cb_avg_mae:.1f}M")
    print(f"  Средний sMAPE: {cb_avg_smape:.1f}%")
    
    print(f"\nLightGBM:")
    print(f"  Средний R²: {lgb_avg_r2:.3f}")
    print(f"  Средний MAE: {lgb_avg_mae:.1f}M")
    print(f"  Средний sMAPE: {lgb_avg_smape:.1f}%")
    
    winner = 'CatBoost' if cb_avg_r2 > lgb_avg_r2 else 'LightGBM'
    improvement = abs(cb_avg_r2 - lgb_avg_r2) / min(cb_avg_r2, lgb_avg_r2) * 100
    
    print(f"\n→ ЛУЧШАЯ МОДЕЛЬ: {winner} (+{improvement:.1f}% по R²)")
else:
    print("Нет результатов для сравнения!")

print("\n" + "="*100)
print("ДЕТАЛЬНЫЕ МЕТРИКИ ПО СЕГМЕНТАМ")
print("="*100)

for segment_id in training_results['catboost'].keys():
    print(f"\n{'='*70}")
    print(f"Сегмент: {segment_id}")
    print(f"{'='*70}")
    
    print("\nCatBoost:")
    for metric, value in training_results['catboost'][segment_id].items():
        if metric not in ['feature_importance', 'train_samples', 'val_samples']:
            print(f"  {metric}: {value}")
    
    print("\nLightGBM:")
    for metric, value in training_results['lightgbm'][segment_id].items():
        if metric not in ['feature_importance', 'train_samples', 'val_samples']:
            print(f"  {metric}: {value}")

print("\n✓ Обучение завершено!")

In [ ]:
# ========== ВИЗУАЛИЗАЦИЯ FEATURE IMPORTANCE ==========

print("\n" + "="*100)
print("TOP-10 ВАЖНЫХ ПРИЗНАКОВ")
print("="*100)

for segment_id in training_results['catboost'].keys():
    print(f"\n{'='*70}")
    print(f"Сегмент: {segment_id}")
    print(f"{'='*70}")
    
    print("\nCatBoost Top-10:")
    cb_importance = catboost_cltv.feature_importance[segment_id].head(10)
    print(cb_importance.to_string(index=False))
    
    print("\nLightGBM Top-10:")
    lgb_importance = lightgbm_cltv.feature_importance[segment_id].head(10)
    print(lgb_importance.to_string(index=False))

# Визуализация для первого сегмента
if len(training_results['catboost']) > 0:
    first_segment = list(training_results['catboost'].keys())[0]
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # CatBoost
    cb_top = catboost_cltv.feature_importance[first_segment].head(15)
    axes[0].barh(cb_top['feature'], cb_top['importance'])
    axes[0].set_xlabel('Importance')
    axes[0].set_title(f'CatBoost Feature Importance ({first_segment})')
    axes[0].invert_yaxis()
    
    # LightGBM
    lgb_top = lightgbm_cltv.feature_importance[first_segment].head(15)
    axes[1].barh(lgb_top['feature'], lgb_top['importance'])
    axes[1].set_xlabel('Importance')
    axes[1].set_title(f'LightGBM Feature Importance ({first_segment})')
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.show()

print("\n✓ Визуализация завершена!")